# 🛡️ Handling Edge Cases in RAG

## Real RAG Systems Break in Predictable Ways

```
❓ "What is the capital of Mars?"  → nothing relevant retrieved
🔀 "Is dropout good or bad?"       → conflicting docs retrieved
📋 "List all 50 employees"         → LLM can't fit all in context
🌍 "Bonjour, comment vas-tu?"       → query in a different language
```

A production RAG system needs to handle all of these gracefully.

## What You'll Learn

1. No relevant docs found → fallback response
2. Low confidence retrieval → acknowledge uncertainty
3. Conflicting information in docs → present both sides
4. Query outside scope → redirect politely

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

# Small knowledge base — only covers ML topics
knowledge_base = [
    "Dropout prevents overfitting by randomly dropping neurons during training.",
    "L2 regularization prevents overfitting by penalizing large weights.",
    "Batch normalization stabilizes training by normalizing layer inputs.",
    "Early stopping halts training when validation loss stops improving.",
]

kb_embeddings = model.encode(knowledge_base)
print(f"Knowledge base: {len(knowledge_base)} docs (ML topics only)")

## 1. Edge Case: No Relevant Docs Found

In [ ]:
# Retrieve with a confidence threshold
# If no doc scores above the threshold, retrieval fails gracefully

CONFIDENCE_THRESHOLD = 0.4  # Below this = not relevant enough

def retrieve_with_threshold(query, threshold=CONFIDENCE_THRESHOLD, top_k=3):
    q_emb = model.encode(query)
    scores = cosine_similarity([q_emb], kb_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Filter by threshold
    results = [
        {"doc": knowledge_base[i], "score": float(scores[i])}
        for i in top_indices
        if scores[i] >= threshold
    ]
    return results


# Test 1: On-topic query
q1 = "How do I prevent overfitting?"
r1 = retrieve_with_threshold(q1)
print(f"Query: '{q1}'")
print(f"Found {len(r1)} relevant docs")
for r in r1:
    print(f"  [{r['score']:.3f}] {r['doc']}")

print()

# Test 2: Off-topic query — nothing relevant in knowledge base
q2 = "What is the population of Tokyo?"
r2 = retrieve_with_threshold(q2)
print(f"Query: '{q2}'")
print(f"Found {len(r2)} relevant docs")

In [ ]:
# Handle the no-results case in generation

def generate_with_fallback(query, retrieved_docs):
    """Generate a response, handling the case where retrieval found nothing."""

    if not retrieved_docs:
        # Graceful fallback — don't hallucinate
        return (
            f"I couldn't find relevant information about '{query}' in the knowledge base. "
            "Please try rephrasing your question or ask about a different topic."
        )

    # Normal case — build prompt and call LLM
    context = "\n".join(f"[{i+1}] {r['doc']}" for i, r in enumerate(retrieved_docs))
    prompt = f"""Answer using only the context below. Cite sources [1], [2] etc.
If unsure, say so.

Context:
{context}

Question: {query}
Answer:"""
    return f"[Would call LLM with this prompt]\n\n{prompt}"


# Show the fallback response
response = generate_with_fallback(q2, r2)
print(f"Response for off-topic query:")
print(response)

## 2. Edge Case: Low Confidence (Uncertain Retrieval)

In [ ]:
# Classify retrieval confidence to adjust how the LLM responds

def classify_confidence(retrieved_docs):
    """Return 'high', 'medium', or 'low' based on top score."""
    if not retrieved_docs:
        return "none"
    top_score = retrieved_docs[0]["score"]
    if top_score >= 0.7:
        return "high"
    elif top_score >= 0.45:
        return "medium"
    else:
        return "low"


# Adjust the prompt based on confidence
def make_prompt_for_confidence(question, context, confidence):
    if confidence == "high":
        instruction = "Answer the question using the context below. Be direct and factual."
    elif confidence == "medium":
        instruction = "Answer using the context, but note that the match may be partial. Acknowledge any uncertainty."
    else:
        instruction = "The context below may not fully answer the question. Provide what you can and acknowledge limitations."

    return f"{instruction}\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"


# Test
test_queries = [
    "How does dropout work?",              # Likely high confidence
    "Is regularization always useful?",    # Likely medium
    "Compare ML to traditional software", # Likely low
]

for q in test_queries:
    results = retrieve_with_threshold(q, threshold=0.2)  # Low threshold to always get something
    conf = classify_confidence(results)
    top_score = results[0]['score'] if results else 0
    print(f"'{q}'")
    print(f"  Top score: {top_score:.3f} → Confidence: {conf}")
    print()

## 3. Edge Case: Conflicting Information in Docs

In [ ]:
# Sometimes retrieved docs contradict each other
# The prompt should tell the LLM how to handle this

conflicting_docs = [
    "[1] Dropout is always beneficial and should be used in every neural network layer.",
    "[2] Dropout should NOT be used in convolutional layers — it hurts performance there.",
    "[3] Dropout works best in fully-connected layers and is less effective in conv layers.",
]

conflict_context = "\n".join(conflicting_docs)
question = "Should I use dropout in convolutional layers?"

# Prompt that handles conflict
conflict_prompt = f"""Answer using only the numbered sources below.
If sources contradict each other, acknowledge the disagreement and present both views.
Do not pick one side without evidence. Cite sources inline.

Sources:
{conflict_context}

Question: {question}
Answer:"""

print(conflict_prompt)

# What a good LLM answer looks like
print("\nExample good answer:")
print("-" * 50)
print(
    "Sources disagree on this point. Source [1] suggests dropout is universally beneficial, "
    "but sources [2] and [3] indicate it should NOT be used in convolutional layers, "
    "as it tends to hurt performance there. The consensus in [2] and [3] is that "
    "dropout is most effective in fully-connected layers."
)

## 4. Edge Case: Out-of-Scope Query

In [ ]:
# Define what your RAG system is scoped to handle
SYSTEM_SCOPE = "machine learning and deep learning topics"

out_of_scope_queries = [
    "What is the weather in New York today?",
    "Write me a poem about autumn",
    "What is the price of Bitcoin?",
]

in_scope_queries = [
    "How does gradient descent work?",
    "What is batch normalization?",
]

# Simple scope check using embedding similarity to scope description
scope_embedding = model.encode(SYSTEM_SCOPE)
SCOPE_THRESHOLD = 0.3

def is_in_scope(query):
    q_emb = model.encode(query)
    score = cosine_similarity([q_emb], [scope_embedding])[0][0]
    return score >= SCOPE_THRESHOLD, float(score)


print("Scope check:")
print(f"System scope: '{SYSTEM_SCOPE}'\n")

for q in out_of_scope_queries + in_scope_queries:
    in_scope, score = is_in_scope(q)
    label = "✅ in scope" if in_scope else "❌ out of scope"
    print(f"  {label} [{score:.3f}] '{q}'")

In [ ]:
# Complete pipeline with all edge case handling

def safe_rag_pipeline(query):
    """
    RAG pipeline that handles all the common edge cases:
    - Out-of-scope query
    - No relevant docs found
    - Low confidence retrieval
    """

    # 1. Scope check
    in_scope, scope_score = is_in_scope(query)
    if not in_scope:
        return f"⚠️  This question is outside my scope ({SYSTEM_SCOPE}). " \
               f"Please ask about those topics instead."

    # 2. Retrieve
    results = retrieve_with_threshold(query, threshold=CONFIDENCE_THRESHOLD)

    # 3. No docs found
    if not results:
        return f"I couldn't find relevant information about '{query}'. " \
               f"Try rephrasing your question."

    # 4. Check confidence
    confidence = classify_confidence(results)

    # 5. Build context and (would) call LLM
    context = "\n".join(f"[{i+1}] {r['doc']}" for i, r in enumerate(results))
    prompt = make_prompt_for_confidence(query, context, confidence)

    return f"[Confidence: {confidence}] Would call LLM with {len(results)} docs retrieved."


# Test
test_cases = [
    "How does dropout work?",                # Normal
    "What is the capital of France?",        # Out of scope
    "Explain quantum entanglement",          # Out of scope
]

for q in test_cases:
    print(f"Query: '{q}'")
    print(f"  → {safe_rag_pipeline(q)}")
    print()

## Key Takeaways 🎯

| Edge Case | Detection | Handling |
|---|---|---|
| No docs found | Empty results after threshold filter | Return "I don't know" message |
| Low confidence | Top score < threshold | Prompt LLM to hedge its answer |
| Conflicting docs | N/A (always check) | Prompt LLM to present both sides |
| Out of scope | Scope similarity score | Redirect with a clear message |

**Golden rule:** Failing gracefully is better than a confident wrong answer.

---
Section 4 complete! Next: **Section 6 — Evaluation** — how do you know if your RAG system is actually good?